In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler


df = pd.read_csv('Vehicle.csv')


X = df.drop(columns=['Class'])
y = df['Class']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.30, 
    random_state=25, 
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:


knn_results = []

for k in range(1, 11):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    score = accuracy_score(y_test, y_pred)
    knn_results.append({'k': k, 'score': score})


df_knn_unscaled = pd.DataFrame(knn_results).sort_values(by='score', ascending=False).reset_index(drop=True)
display(df_knn_unscaled)

In [ ]:



scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)


knn_minmax = KNeighborsClassifier(n_neighbors=2)
knn_minmax.fit(X_train_minmax, y_train)


y_pred_minmax = knn_minmax.predict(X_test_minmax)
print("=== MinMaxScaler KNN (k=2) Classification Report ===")
print(classification_report(y_test, y_pred_minmax))

In [ ]:



scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)


knn_std = KNeighborsClassifier(n_neighbors=3)
knn_std.fit(X_train_std, y_train)


y_pred_std = knn_std.predict(X_test_std)
print("=== StandardScaler KNN (k=3) Classification Report ===")
print(classification_report(y_test, y_pred_std))

In [ ]:



lr_default = LogisticRegression(max_iter=3000, random_state=25)
lr_default.fit(X_train, y_train)


y_pred_lr_default = lr_default.predict(X_test)
default_lr_accuracy = accuracy_score(y_test, y_pred_lr_default)

print(f"Default Logistic Regression Accuracy (Unscaled): {default_lr_accuracy:.4f}")

In [ ]:
solvers = ['sag', 'newton-cg', 'newton-cholesky', 'lbfgs']
penalties = ['l2', None]

lr_results = []

for solver in solvers:
    for penalty in penalties:
        try:
       
            model = LogisticRegression(
                solver=solver, 
                penalty=penalty, 
                max_iter=5000, 
                random_state=25
            )
            model.fit(X_train, y_train)
            
           
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            
            lr_results.append({
                'penalty': str(penalty),
                'solver': solver,
                'accuracy': acc
            })
        except Exception as e:
          
            print(f"Skipped combination (solver: {solver}, penalty: {penalty}): {e}")


df_lr_comparison = pd.DataFrame(lr_results).sort_values(by='accuracy', ascending=False).reset_index(drop=True)
display(df_lr_comparison)